# AI Cybersecurity Threat Detection System
# End-to-End ML Pipeline Demonstration

In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv("../data/raw/UNSW_NB15_training-set.csv")

print("Shape:", df.shape)
df.head()

In [ ]:
df = df.drop(columns=['id'], errors='ignore')
df = df.dropna()

print("After cleaning:", df.shape)

In [ ]:
df['byte_ratio'] = df['sbytes'] / (df['dbytes'] + 1)

In [ ]:
X = df.drop(columns=['label', 'attack_cat'], errors='ignore')
y = df['label']

In [ ]:
cat_cols = ['proto', 'service', 'state']

encoders = {}

for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    encoders[col] = le

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42
)

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

In [ ]:
preds = rf_model.predict(X_test)

In [ ]:
print(classification_report(y_test, preds))

In [ ]:
cm = confusion_matrix(y_test, preds)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d')
plt.title("Confusion Matrix")
plt.show()

In [ ]:
iso_model = IsolationForest(contamination=0.15, random_state=42)
iso_model.fit(X_train)

iso_preds = iso_model.predict(X_test)
iso_preds = [1 if p == -1 else 0 for p in iso_preds]

In [ ]:
print(classification_report(y_test, iso_preds))

In [ ]:
plt.figure()
sns.countplot(x=iso_preds)
plt.title("Anomaly Detection Distribution")
plt.show()